## Grouped Subject-Aware Modeling and Uncertainty

This notebook rebuilds evaluation to be robust and subject-aware while staying lean.

Scope remains unchanged:
1. heart-rate forecasting (regression),
2. current activity classification,
3. uncertainty intervals for regression.

Evaluation strategy in this notebook:
- leave-one-subject-out cross-validation (LOSO) for model comparison and selection,
- compact model roster only (simple baselines plus one tree model per task),
- explicit fold, subject, and activity breakdown reporting,
- grouped conformal intervals using subject-disjoint train, calibration, and test segments.

All outputs are saved to `artifacts/metrics/` and `artifacts/figures/` with `grouped_cv_` prefixes.

In [1]:
from pathlib import Path
import sys

import pandas as pd

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Centralized grouped evaluation helper keeps notebook logic concise.
from scripts.grouped_evaluation import ALPHA, RANDOM_SEED, run_grouped_evaluation

PROCESSED_PATH = REPO_ROOT / "data" / "processed" / "pamap2_model_table.parquet"
METRICS_DIR = REPO_ROOT / "artifacts" / "metrics"
FIGURES_DIR = REPO_ROOT / "artifacts" / "figures"
MODELS_DIR = REPO_ROOT / "artifacts" / "models"

print(f"Repo root: {REPO_ROOT}")
print(f"Processed table exists: {PROCESSED_PATH.exists()}")
print(f"Using random seed: {RANDOM_SEED}")
print(f"Conformal alpha: {ALPHA}")

Repo root: G:\Other computers\My Mac\Documents\pamap2_telemetry\pamap2_telemetry
Processed table exists: True
Using random seed: 42
Conformal alpha: 0.1


In [2]:
if not PROCESSED_PATH.exists():
    raise FileNotFoundError(f"Missing processed model table: {PROCESSED_PATH}")

model_df = pd.read_parquet(PROCESSED_PATH).copy()
model_df = model_df.sort_values(["subject_id", "timestamp_s"]).reset_index(drop=True)

required_columns = [
    "subject_id",
    "timestamp_s",
    "activity_target",
    "activity_label",
    "heart_rate_bpm",
    "hr_target_30s",
]
missing_required = [column for column in required_columns if column not in model_df.columns]
if missing_required:
    raise ValueError(f"Processed table is missing required columns: {missing_required}")

duplicate_count = int(model_df.duplicated(subset=["subject_id", "timestamp_s"]).sum())
if duplicate_count > 0:
    raise ValueError(f"Found duplicate subject-second rows: {duplicate_count}")

print(f"Rows: {len(model_df):,}")
print(f"Columns: {len(model_df.columns)}")
print(f"Subjects: {sorted(model_df['subject_id'].unique().tolist())}")
display(model_df.head())

Rows: 18,627
Columns: 63
Subjects: [101, 102, 103, 104, 105, 106, 107, 108]


,subject_id,session,timestamp_s,activity_id,activity_label,heart_rate_bpm,hand_acc_16g_mag,chest_acc_16g_mag,ankle_acc_16g_mag,hand_gyro_mag,...,chest_gyro_mag_delta_from_rollmean_5,ankle_gyro_mag_lag_1,ankle_gyro_mag_lag_5,ankle_gyro_mag_rollmean_5,ankle_gyro_mag_rollstd_5,ankle_gyro_mag_rollmean_10,ankle_gyro_mag_rollstd_10,ankle_gyro_mag_delta_from_rollmean_5,hr_target_30s,activity_target
0,101,protocol,46,1,lying,102.111111,9.828055,9.525052,9.937526,2.361510,...,0.414428,0.222166,0.042753,0.139954,0.138818,0.095260,0.104996,0.206270,94.555556,1
1,101,protocol,47,1,lying,103.000000,10.387578,10.717709,10.854565,1.594349,...,0.783349,0.346223,0.048473,0.231953,0.201368,0.142722,0.164539,0.276514,94.000000,1
2,101,protocol,48,1,lying,103.000000,10.297232,9.662208,10.442077,1.276431,...,0.174163,0.508467,0.040405,0.672534,0.894499,0.363615,0.679581,1.570779,94.000000,1
3,101,protocol,49,1,lying,103.555556,9.832847,9.867935,10.209344,0.599974,...,-0.135288,2.243313,0.042502,0.723322,0.856172,0.384258,0.673454,-0.426882,94.000000,1
4,101,protocol,50,1,lying,104.000000,9.815614,9.695050,10.096223,0.548152,...,-0.379189,0.296440,0.222166,0.764309,0.830710,0.421784,0.663250,-0.337207,94.000000,1


## Run Grouped LOSO Evaluation

This cell executes the full grouped workflow and writes all metric tables and figures used for model selection and error-breakdown reporting.

In [3]:
results = run_grouped_evaluation(
    processed_path=PROCESSED_PATH,
    metrics_dir=METRICS_DIR,
    figures_dir=FIGURES_DIR,
    models_dir=MODELS_DIR,
    random_seed=RANDOM_SEED,
    alpha=ALPHA,
)

print("Grouped evaluation complete.")
print("Saved grouped metrics and figures with grouped_cv_ file names.")
print("Returned result keys:", sorted(results.keys()))

Grouped evaluation complete.
Saved grouped metrics and figures with grouped_cv_ file names.
Returned result keys: ['classification_by_activity', 'classification_by_subject', 'classification_fold', 'classification_per_class', 'classification_summary', 'conformal_by_activity', 'conformal_by_subject', 'conformal_fold', 'conformal_summary', 'regression_by_activity', 'regression_by_subject', 'regression_fold', 'regression_summary', 'selected_models']


In [4]:
regression_fold_df = results["regression_fold"].copy()
regression_summary_df = results["regression_summary"].copy()

print("Regression fold-level metrics:")
display(regression_fold_df.sort_values(["model", "fold"]).reset_index(drop=True))

print("Regression grouped CV summary (mean/std/min/max across folds):")
display(regression_summary_df.sort_values("rank"))

Regression fold-level metrics:


,task,model,fold,test_subject_id,train_subject_count,mae,rmse,r2
0,regression,hist_gradient_boosting,1,101,7,7.961057,10.498332,0.822087
1,regression,hist_gradient_boosting,2,102,7,5.951150,9.213178,0.820979
2,regression,hist_gradient_boosting,3,103,7,4.517481,6.302510,0.918660
3,regression,hist_gradient_boosting,4,104,7,7.865632,10.955475,0.720735
4,regression,hist_gradient_boosting,5,105,7,5.666353,8.137151,0.894868
5,regression,hist_gradient_boosting,6,106,7,7.649216,10.601356,0.857292
6,regression,hist_gradient_boosting,7,107,7,8.707860,12.549291,0.729360
7,regression,hist_gradient_boosting,8,108,7,6.910758,10.258927,0.870316
8,regression,linear_regression,1,101,7,7.234359,9.891742,0.842052
9,regression,linear_regression,2,102,7,6.158688,9.724433,0.800559


Regression grouped CV summary (mean/std/min/max across folds):


,task,model,fold_count,mean_mae,std_mae,min_mae,max_mae,mean_rmse,std_rmse,min_rmse,max_rmse,mean_r2,std_r2,min_r2,max_r2,rank,delta_mean_mae_vs_best
0,regression,hist_gradient_boosting,8,6.903688,1.413535,4.517481,8.707860,9.814528,1.911407,6.302510,12.549291,0.829287,0.072354,0.720735,0.918660,1,0.000000
1,regression,linear_regression,8,7.157205,0.981724,6.078320,8.474256,10.274584,1.389972,8.180096,12.061424,0.814994,0.064413,0.692778,0.877095,2,0.253517
2,regression,persistence_current_hr,8,7.220209,1.015102,5.728746,8.728120,11.425830,1.604433,9.554996,13.919609,0.773276,0.070812,0.664562,0.848975,3,0.316521


In [5]:
classification_fold_df = results["classification_fold"].copy()
classification_summary_df = results["classification_summary"].copy()
classification_per_class_df = results["classification_per_class"].copy()

print("Classification fold-level metrics:")
display(classification_fold_df.sort_values(["model", "fold"]).reset_index(drop=True))

print("Classification grouped CV summary (mean/std/min/max across folds):")
display(classification_summary_df.sort_values("rank"))

print("Selected classification model per-class performance:")
display(classification_per_class_df.sort_values("support", ascending=False).reset_index(drop=True))

Classification fold-level metrics:


,task,model,fold,test_subject_id,train_subject_count,accuracy,macro_f1
0,classification,logistic_regression,1,101,7,0.724153,0.719915
1,classification,logistic_regression,2,102,7,0.734661,0.701714
2,classification,logistic_regression,3,103,7,0.800469,0.626305
3,classification,logistic_regression,4,104,7,0.819060,0.749408
4,classification,logistic_regression,5,105,7,0.779056,0.776060
5,classification,logistic_regression,6,106,7,0.771231,0.757626
6,classification,logistic_regression,7,107,7,0.754256,0.640200
7,classification,logistic_regression,8,108,7,0.856112,0.842490
8,classification,random_forest,1,101,7,0.696697,0.684164
9,classification,random_forest,2,102,7,0.728566,0.732248


Classification grouped CV summary (mean/std/min/max across folds):


,task,model,fold_count,mean_macro_f1,std_macro_f1,min_macro_f1,max_macro_f1,mean_accuracy,std_accuracy,min_accuracy,max_accuracy,rank,delta_mean_macro_f1_vs_best
0,classification,random_forest,8,0.745343,0.071226,0.635825,0.825856,0.794325,0.058579,0.696697,0.872653,1,0.000000
1,classification,logistic_regression,8,0.726715,0.071262,0.626305,0.842490,0.779875,0.044163,0.724153,0.856112,2,0.018628


Selected classification model per-class performance:


,activity_target,activity_label,precision,recall,f1,support
0,17,ironing,0.817068,0.866359,0.840992,2387
1,4,walking,0.787310,0.835309,0.810600,2362
2,3,standing,0.564919,0.476818,0.517143,1898
3,7,nordic_walking,0.805369,0.701382,0.749787,1882
4,1,lying,0.956240,0.955208,0.955724,1853
5,2,sitting,0.523855,0.598489,0.558690,1853
6,16,vacuum_cleaning,0.845422,0.811181,0.827948,1753
7,6,cycling,0.838218,0.884354,0.860668,1617
8,12,ascending_stairs,0.889932,0.889932,0.889932,1172
9,13,descending_stairs,0.926570,0.915950,0.921230,1047


## Final Model Selection and Breakdown Reporting

Selection is driven by grouped CV summaries only, not one validation subject.

Rules used:
- Regression: lowest mean MAE across LOSO folds, tie-break by MAE standard deviation then mean RMSE.
- Classification: highest mean macro F1 across LOSO folds, tie-break by macro F1 standard deviation then mean accuracy.

In [6]:
selected_models_df = results["selected_models"].copy()

regression_by_subject_df = results["regression_by_subject"].copy()
regression_by_activity_df = results["regression_by_activity"].copy()
classification_by_subject_df = results["classification_by_subject"].copy()
classification_by_activity_df = results["classification_by_activity"].copy()

print("Selected-model summary table:")
display(selected_models_df)

print("Regression performance by subject:")
display(regression_by_subject_df.sort_values("subject_id").reset_index(drop=True))

print("Regression performance by activity:")
display(regression_by_activity_df.sort_values("rows", ascending=False).reset_index(drop=True))

print("Classification performance by subject:")
display(classification_by_subject_df.sort_values("subject_id").reset_index(drop=True))

print("Classification performance by activity:")
display(classification_by_activity_df.sort_values("rows", ascending=False).reset_index(drop=True))

Selected-model summary table:


,task,selected_model,selection_rule,selected_mean_mae,selected_std_mae,runner_up_model,runner_up_mean_mae,winner_margin,selected_mean_macro_f1,selected_std_macro_f1,runner_up_mean_macro_f1
0,regression,hist_gradient_boosting,"lowest mean MAE across LOSO folds, tie-break b...",6.903688,1.413535,linear_regression,7.157205,0.253517,NaN,NaN,NaN
1,classification,random_forest,"highest mean macro F1 across LOSO folds, tie-b...",NaN,NaN,logistic_regression,NaN,0.018628,0.745343,0.071226,0.726715


Regression performance by subject:


,subject_id,rows,mae,rmse,r2,mean_abs_error
0,101,2331,7.961057,10.498332,0.822087,7.961057
1,102,2461,5.951150,9.213178,0.820979,5.951150
2,103,1704,4.517481,6.302510,0.918660,4.517481
3,104,2277,7.865632,10.955475,0.720735,7.865632
4,105,2607,5.666353,8.137151,0.894868,5.666353
5,106,2461,7.649216,10.601356,0.857292,7.649216
6,107,2291,8.707860,12.549291,0.729360,8.707860
7,108,2495,6.910758,10.258927,0.870316,6.910758


Regression performance by activity:


,activity_target,activity_label,rows,mae,rmse,r2,mean_abs_error
0,17,ironing,2387,4.854663,6.376949,0.513126,4.854663
1,4,walking,2362,4.221345,5.660409,0.667658,4.221345
2,3,standing,1898,5.378603,6.989465,0.519034,5.378603
3,7,nordic_walking,1882,6.279882,8.613937,0.289370,6.279882
4,1,lying,1853,8.940536,12.319956,-0.502343,8.940536
5,2,sitting,1853,6.483666,9.007480,-0.254845,6.483666
6,16,vacuum_cleaning,1753,5.798676,7.560648,0.666054,5.798676
7,6,cycling,1617,5.939949,8.239981,0.038593,5.939949
8,12,ascending_stairs,1172,18.362604,22.270715,-0.183654,18.362604
9,13,descending_stairs,1047,9.935479,12.834340,0.378260,9.935479


Classification performance by subject:


,subject_id,rows,accuracy,macro_f1
0,101,2331,0.696697,0.684164
1,102,2461,0.728566,0.732248
2,103,1704,0.872653,0.635825
3,104,2277,0.830040,0.760333
4,105,2607,0.815497,0.812529
5,106,2461,0.828931,0.825856
6,107,2291,0.767787,0.692022
7,108,2495,0.814429,0.819768


Classification performance by activity:


,activity_target,activity_label,rows,accuracy,macro_f1
0,17,ironing,2387,0.866359,0.084400
1,4,walking,2362,0.835309,0.082751
2,3,standing,1898,0.476818,0.058703
3,7,nordic_walking,1882,0.701382,0.074953
4,1,lying,1853,0.955208,0.088826
5,2,sitting,1853,0.598489,0.068074
6,16,vacuum_cleaning,1753,0.811181,0.081432
7,6,cycling,1617,0.884354,0.085330
8,12,ascending_stairs,1172,0.889932,0.085615
9,13,descending_stairs,1047,0.915950,0.086921


## Leakage and Coverage Checks

This section confirms fold-wise subject isolation and summarizes grouped conformal interval quality.

In [7]:
subject_count = int(model_df["subject_id"].nunique())

reg_subject_coverage = (
    regression_fold_df.groupby(["model", "test_subject_id"], as_index=False)
    .size()
    .rename(columns={"size": "row_count"})
)
cls_subject_coverage = (
    classification_fold_df.groupby(["model", "test_subject_id"], as_index=False)
    .size()
    .rename(columns={"size": "row_count"})
)

reg_subjects_per_model = regression_fold_df.groupby("model")["test_subject_id"].nunique()
cls_subjects_per_model = classification_fold_df.groupby("model")["test_subject_id"].nunique()

if not (reg_subjects_per_model == subject_count).all():
    raise ValueError("Regression grouped CV did not cover each subject exactly once per model.")
if not (cls_subjects_per_model == subject_count).all():
    raise ValueError("Classification grouped CV did not cover each subject exactly once per model.")

print("Leakage guard checks passed.")
print(f"Unique subjects in modeling table: {subject_count}")
print("Regression subjects per model:")
display(reg_subjects_per_model.rename("unique_test_subjects"))
print("Classification subjects per model:")
display(cls_subjects_per_model.rename("unique_test_subjects"))

Leakage guard checks passed.
Unique subjects in modeling table: 8
Regression subjects per model:


model
hist_gradient_boosting    8
linear_regression         8
persistence_current_hr    8
Name: unique_test_subjects, dtype: int64

Classification subjects per model:


model
logistic_regression    8
random_forest          8
Name: unique_test_subjects, dtype: int64

## Grouped Conformal Uncertainty

Conformal intervals are computed fold by fold with disjoint subject sets:
- proper-train subjects fit the selected regression model,
- one separate calibration subject sets the conformal margin,
- the held-out test subject receives interval predictions.

This preserves subject isolation during uncertainty evaluation.

In [8]:
conformal_fold_df = results["conformal_fold"].copy()
conformal_summary_df = results["conformal_summary"].copy()
conformal_by_subject_df = results["conformal_by_subject"].copy()
conformal_by_activity_df = results["conformal_by_activity"].copy()

print("Conformal fold summary:")
display(conformal_fold_df.sort_values("fold").reset_index(drop=True))

print("Conformal aggregate summary:")
display(conformal_summary_df)

print("Conformal coverage by subject:")
display(conformal_by_subject_df.sort_values("subject_id").reset_index(drop=True))

print("Conformal coverage by activity:")
display(conformal_by_activity_df.sort_values("rows", ascending=False).reset_index(drop=True))

Conformal fold summary:


,fold,test_subject_id,calibration_subject_id,proper_train_subject_count,n_proper_train,n_calibration,n_test,alpha,target_coverage,q_hat,empirical_coverage,average_interval_width
0,1,101,108,6,13801,2495,2331,0.1,0.9,18.053623,0.914200,36.107247
1,2,102,108,6,13671,2495,2461,0.1,0.9,18.171552,0.957741,36.343104
2,3,103,108,6,14428,2495,1704,0.1,0.9,17.826718,0.964202,35.653436
3,4,104,108,6,13855,2495,2277,0.1,0.9,18.285569,0.906017,36.571139
4,5,105,108,6,13525,2495,2607,0.1,0.9,18.447349,0.957806,36.894698
5,6,106,108,6,13671,2495,2461,0.1,0.9,19.293302,0.932548,38.586604
6,7,107,108,6,13841,2495,2291,0.1,0.9,18.338234,0.903099,36.676468
7,8,108,107,6,13841,2291,2495,0.1,0.9,17.866659,0.897796,35.733318


Conformal aggregate summary:


,selected_regression_model,alpha,target_coverage,mean_fold_coverage,std_fold_coverage,min_fold_coverage,max_fold_coverage,mean_fold_interval_width,std_fold_interval_width,row_level_empirical_coverage,row_level_average_interval_width,fold_count,n_predictions
0,hist_gradient_boosting,0.1,0.9,0.929176,0.027514,0.897796,0.964202,36.570752,0.924987,0.928491,36.611308,8,18627


Conformal coverage by subject:


,subject_id,rows,empirical_coverage,average_interval_width
0,101,2331,0.914200,36.107247
1,102,2461,0.957741,36.343104
2,103,1704,0.964202,35.653436
3,104,2277,0.906017,36.571139
4,105,2607,0.957806,36.894698
5,106,2461,0.932548,38.586604
6,107,2291,0.903099,36.676468
7,108,2495,0.897796,35.733318


Conformal coverage by activity:


,activity_target,activity_label,rows,empirical_coverage,average_interval_width
0,17,ironing,2387,0.984499,36.651528
1,4,walking,2362,0.986876,36.561431
2,3,standing,1898,0.979452,36.587459
3,7,nordic_walking,1882,0.941552,36.703798
4,1,lying,1853,0.944954,36.566340
5,2,sitting,1853,0.948192,36.543280
6,16,vacuum_cleaning,1753,0.962350,36.561484
7,6,cycling,1617,0.959184,36.655959
8,12,ascending_stairs,1172,0.529863,36.592250
9,13,descending_stairs,1047,0.833811,36.530408


In [9]:
grouped_metric_files = sorted(METRICS_DIR.glob("grouped_cv_*.csv"))
grouped_figure_files = sorted(FIGURES_DIR.glob("grouped_cv_*.png"))

print("Grouped metric artifacts:")
for path in grouped_metric_files:
    print(path.relative_to(REPO_ROOT))

print("\nGrouped figure artifacts:")
for path in grouped_figure_files:
    print(path.relative_to(REPO_ROOT))

Grouped metric artifacts:
artifacts\metrics\grouped_cv_classification_fold_metrics.csv
artifacts\metrics\grouped_cv_classification_predictions_all_models.csv
artifacts\metrics\grouped_cv_classification_selected_by_activity.csv
artifacts\metrics\grouped_cv_classification_selected_by_subject.csv
artifacts\metrics\grouped_cv_classification_selected_per_class.csv
artifacts\metrics\grouped_cv_classification_summary.csv
artifacts\metrics\grouped_cv_conformal_by_activity.csv
artifacts\metrics\grouped_cv_conformal_by_subject.csv
artifacts\metrics\grouped_cv_conformal_fold_summary.csv
artifacts\metrics\grouped_cv_conformal_predictions.csv
artifacts\metrics\grouped_cv_conformal_summary.csv
artifacts\metrics\grouped_cv_regression_fold_metrics.csv
artifacts\metrics\grouped_cv_regression_predictions_all_models.csv
artifacts\metrics\grouped_cv_regression_selected_by_activity.csv
artifacts\metrics\grouped_cv_regression_selected_by_subject.csv
artifacts\metrics\grouped_cv_regression_summary.csv
artifa

## Notes on Compactness and Defensibility

This workflow intentionally keeps model comparison small and interpretable.

Regression models compared:
- persistence baseline (current heart rate),
- linear regression,
- histogram gradient boosting regressor.

Classification models compared:
- logistic regression,
- random forest classifier.

The final story now answers:
- how models perform across all held-out subjects,
- where errors concentrate by subject and activity,
- whether simple baselines are already competitive.

In [10]:
# Compact interview-oriented summary pulled from grouped evaluation outputs.
reg_choice = selected_models_df[selected_models_df["task"] == "regression"].iloc[0]
cls_choice = selected_models_df[selected_models_df["task"] == "classification"].iloc[0]

print("Final model choices from grouped CV:")
print(
    f"Regression: {reg_choice['selected_model']} | mean MAE={reg_choice['selected_mean_mae']:.3f} "
    f"(runner-up margin={reg_choice['winner_margin']:.3f})"
)
print(
    f"Classification: {cls_choice['selected_model']} | mean macro F1={cls_choice['selected_mean_macro_f1']:.3f} "
    f"(runner-up margin={cls_choice['winner_margin']:.3f})"
)

print("\nKey risk surfaces to discuss:")
print("- Highest-regression-error activities:")
display(
    regression_by_activity_df.sort_values("mae", ascending=False).head(5)[
        ["activity_label", "rows", "mae", "rmse", "r2"]
    ]
)
print("- Lowest-classification-F1 classes:")
display(
    classification_per_class_df.sort_values("f1", ascending=True).head(5)[
        ["activity_label", "support", "precision", "recall", "f1"]
    ]
)

Final model choices from grouped CV:
Regression: hist_gradient_boosting | mean MAE=6.904 (runner-up margin=0.254)
Classification: random_forest | mean macro F1=0.745 (runner-up margin=0.019)

Key risk surfaces to discuss:
- Highest-regression-error activities:


,activity_label,rows,mae,rmse,r2
8,ascending_stairs,1172,18.362604,22.270715,-0.183654
9,descending_stairs,1047,9.935479,12.834340,0.378260
4,lying,1853,8.940536,12.319956,-0.502343
10,running,803,7.147104,9.913377,0.447189
5,sitting,1853,6.483666,9.007480,-0.254845


- Lowest-classification-F1 classes:


,activity_label,support,precision,recall,f1
2,standing,1898,0.564919,0.476818,0.517143
5,sitting,1853,0.523855,0.598489,0.558690
3,nordic_walking,1882,0.805369,0.701382,0.749787
1,walking,2362,0.787310,0.835309,0.810600
6,vacuum_cleaning,1753,0.845422,0.811181,0.827948


## Completion Checklist

This notebook now satisfies the evaluation rebuild goals:
- subject-grouped model selection using LOSO cross-validation,
- compact model roster for both tasks,
- fold-level and aggregate model comparison outputs,
- explicit selected-model summary with selection criteria,
- by-subject and by-activity breakdown tables,
- per-class classification performance,
- grouped conformal uncertainty reporting.

All non-obvious evaluation changes are documented in project docs and decision log.